# nrc-ai Quickstart

Welcome! This notebook walks you through the **Nexus Resonance Codex AI Enhancements** (`nrc-ai`) — drop-in PyTorch modules that replace standard stochastic heuristics with golden-ratio ($\phi$) geometry, modular residue exclusion (TUPT), and Lyapunov-bounded stability (MST / TTT-7).

**Run it anywhere:** click *Open in GitHub Codespaces* on the repo README for a 1-click cloud environment, or run locally after `pip install -e "."`.

Each section below has a short explanation followed by code you can run and tweak.

## 1. Imports

We import PyTorch and five representative `nrc-ai` modules: a normalizer, an attention layer, a rotary embedding, a LoRA adapter, and a learning-rate scheduler.

In [1]:
import torch
from nrc_ai import (
    GoldenFlowNorm,
    GoldenSpiralRotaryEmbedding,
    HodgePhiTTorsionAttention,
    PhiInfinityLosslessLoRA,
    PisanoModulatedLRSchedule,
)

dim, heads = 512, 8
print(f"torch {torch.__version__} | working with dim={dim}, heads={heads}")

torch 2.14.0+cpu | working with dim=512, heads=8


## 2. Attention stack: norm → rotary positions → φ-torsion attention

`GoldenFlowNorm` is a drop-in normalization layer. `GoldenSpiralRotaryEmbedding` applies golden-spiral ($137.5^\circ$) rotary positions — note it expects 4-D input `(batch, heads, seq_len, head_dim)` with `seq_dim=2`. `HodgePhiTTorsionAttention` is the stabilized attention layer.

In [2]:
# Drop-in NRC replacements for standard transformer components
norm = GoldenFlowNorm(hidden_dim=dim)
attn = HodgePhiTTorsionAttention(embed_dim=dim, num_heads=heads)
rope = GoldenSpiralRotaryEmbedding(dim=dim // heads)

# A single forward pass through attention
x = torch.randn(2, 64, dim)                  # (batch, seq_len, embed_dim)
q = torch.randn(2, heads, 64, dim // heads)  # (batch, heads, seq_len, head_dim)
q_rot = rope(q, seq_dim=2)                   # golden-spiral rotary positions
out = attn(norm(x))                          # φ-torsion stabilized attention
print("attention output:", out.shape)        # torch.Size([2, 64, 512])

attention output: torch.Size([2, 64, 512])


## 3. φ-lossless LoRA adapter

`PhiInfinityLosslessLoRA` wraps any linear layer with a fractal low-rank adapter for parameter-efficient fine-tuning.

In [3]:
adapter = PhiInfinityLosslessLoRA(dim, dim, rank=8)
y = adapter(torch.randn(2, 64, dim))
print("adapter output:", y.shape)  # torch.Size([2, 64, 512])

adapter output: torch.Size([2, 64, 512])


## 4. Pisano-modulated learning-rate schedule

`PisanoModulatedLRSchedule` modulates the learning rate on Pisano-period cycles. It wraps any PyTorch optimizer and steps just like the built-in schedulers.

In [4]:
param = torch.nn.Parameter(torch.randn(dim))
opt = torch.optim.AdamW([param], lr=1e-3)
sched = PisanoModulatedLRSchedule(opt)

opt.zero_grad()
param.sum().backward()
opt.step()
sched.step()
print("learning rate after one step:", sched.get_last_lr())

learning rate after one step: [0.001600996901894429]


## Next steps

- Browse all 32 modules in the [README](https://github.com/Nexus-Resonance-Codex/Ai-Enhancements#architectural-taxonomy).
- Read the module guides in the [wiki](https://github.com/Nexus-Resonance-Codex/Ai-Enhancements/wiki).
- Try the 14 expert prompts in [`.github/prompts/`](https://github.com/Nexus-Resonance-Codex/Ai-Enhancements/tree/main/.github/prompts) with Copilot Chat.
- Run the test suite: `pytest tests/`